# Day 3: Classification Algorithms, Model Selection, Classification Evaluation Metrics


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report

# Load Dataset (Breast Cancer Wisconsin)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

X.head()

### Data Splitting and Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Model 1: Logistic Regression

In [ ]:
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_pred_log = log_reg.predict(X_test_scaled)
y_proba_log = log_reg.predict_proba(X_test_scaled)[:, 1]

### Model 2: Decision Tree Classifier

In [ ]:
tree_clf = DecisionTreeClassifier(random_state=42)
tree_clf.fit(X_train_scaled, y_train)
y_pred_tree = tree_clf.predict(X_test_scaled)
y_proba_tree = tree_clf.predict_proba(X_test_scaled)[:, 1]

### Model 3: Random Forest Classifier

In [ ]:
rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train_scaled, y_train)
y_pred_rf = rf_clf.predict(X_test_scaled)
y_proba_rf = rf_clf.predict_proba(X_test_scaled)[:, 1]

### Model Evaluation & Comparison

In [ ]:
def evaluate_model(y_true, y_pred, y_proba, model_name):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba)
    }

results = []
results.append(evaluate_model(y_test, y_pred_log, y_proba_log, 'Logistic Regression'))
results.append(evaluate_model(y_test, y_pred_tree, y_proba_tree, 'Decision Tree'))
results.append(evaluate_model(y_test, y_pred_rf, y_proba_rf, 'Random Forest'))

results_df = pd.DataFrame(results).set_index('Model')
display(results_df)

### Confusion Matrix for the Best Model

In [ ]:
# Identify the best model based on F1-score
best_model_name = results_df['F1-score'].idxmax()
print(f"Best Model based on F1-score: {best_model_name}\n")

# Map the name to the actual predictions
pred_map = {
    'Logistic Regression': y_pred_log,
    'Decision Tree': y_pred_tree,
    'Random Forest': y_pred_rf
}
best_y_pred = pred_map[best_model_name]

# Plot Confusion Matrix
cm = confusion_matrix(y_test, best_y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=data.target_names, yticklabels=data.target_names)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()